# Optical Character Recognition (OCR) tests

**requisites**
pip install -U keras jax jaxlib  # plus GPU wheels if you want

set it in ~/.keras/keras.json:
    json
    {
    "backend": "jax"
    }


**Build the CRNN+CTC model in Keras**
Keras has an official OCR example showing CNN + RNN + CTC implemented with the Functional API and a custom CTCLayer. The same pattern works on JAX.

Key ideas (high level):

    Input: image tensor (height, width, channels).

    CNN: stack Conv2D + MaxPooling2D layers to reduce height to 1 and compress width.

    Reshape: turn the feature map into a time sequence along the width axis using layers.Reshape or layers.Permute.

    RNN: Bidirectional(layers.LSTM(..., return_sequences=True)) for sequence modeling.

    Output: Dense(num_classes + 1, activation="softmax") for per‑time‑step character probabilities.

    CTC: custom layers.Layer that calls keras.backend.ctc_batch_cost in call() and uses self.add_loss(...) so model.fit() trains end‑to‑end.

You can adapt the OCR CAPTCHA example directly; it already defines a CRNN‑style architecture and CTC loss layer.


## pipeline architecture
Input Image → Text Detection (EAST/CRAFT) → Crop Text Lines → CRNN Recognition → Language Model Correction → Final Text


## Step 1: Text Detection with EAST (line-level boxes)

First, detect text regions. EAST is perfect because it outputs word/line-level bounding boxes directly.

Download EAST model: frozen_east_text_detection.pb from https://github.com/argman/EAST

In [ ]:
import cv2
import numpy as np
from typing import List, Tuple


In [ ]:

class EASTDetector:
    def __init__(self, model_path: str):
        self.net = cv2.dnn.readNet(model_path)
    
    def detect(self, image: np.ndarray) -> List[Tuple[int, int, int, int]]:
        """Returns list of (x1,y1,x2,y2) text line boxes"""
        h, w = image.shape[:2]
        blob = cv2.dnn.blobFromImage(image, 1.0, (320, 320), (123.68, 116.78, 103.94), swapRB=True, crop=False)
        
        self.net.setInput(blob)
        scores, geometry = self.net.forward(["feature_fusion/Conv_7/Sigmoid", "feature_fusion/concat_3"])
        
        boxes = []
        for y in range(scores.shape[2]):
            scores_data = scores[0, 0, y]
            x0, y0, x1, y1 = geometry[0, 0, y], geometry[0, 1, y], geometry[0, 2, y], geometry[0, 3, y]
            angle = geometry[0, 4, y]
            
            for x in range(scores_data.shape[1]):
                if scores_data[x] > 0.5:  # confidence threshold
                    offset_x, offset_y = x * 4.0, y * 4.0
                    angle_rad = angle[x][y]
                    
                    # Decode rotated box (simplified to axis-aligned for CRNN)
                    x1, y1 = int(offset_x + x0[x][y] * 4), int(offset_y + y0[x][y] * 4)
                    x2, y2 = int(offset_x + x1[x][y] * 4), int(offset_y + y1[x][y] * 4)
                    boxes.append((x1, y1, x2, y2))
        
        return self.filter_boxes(boxes, image.shape)
    
    def filter_boxes(self, boxes, shape):
        """NMS + area filter"""
        if not boxes:
            return []
        boxes = np.array(boxes)
        indices = cv2.dnn.NMSBoxes(boxes.tolist(), np.ones(len(boxes)), 0.5, 0.4)
        if len(indices) > 0:
            return boxes[indices.flatten()].tolist()
        return []


## Step 2: CRNN Text Recognition (from previous discussion)

Use the Keras CRNN model we discussed. Here's the complete version:

import tensorflow as tf
from tensorflow import keras
import numpy as np

class CRNNRecognizer:
    def __init__(self, model_path: str, char_to_idx: dict):
        self.model = keras.models.load_model(model_path)
        self.char_to_idx = char_to_idx
        self.idx_to_char = {i: c for c, i in char_to_idx.items()}
        self.blank_idx = 0
    
    def predict(self, cropped_lines: List[np.ndarray]) -> List[str]:
        """Process batch of cropped text line images"""
        batch = self.preprocess_lines(cropped_lines)
        predictions = self.model(batch)
        return self.ctc_decode(predictions)
    
    def preprocess_lines(self, lines: List[np.ndarray]) -> np.ndarray:
        """Resize to 32px height, normalize"""
        processed = []
        for line in lines:
            h, w = line.shape[:2]
            new_w = int(w * 32 / h)
            line = cv2.resize(line, (new_w, 32))
            line = cv2.cvtColor(line, cv2.COLOR_BGR2GRAY) / 255.0
            line = line[..., np.newaxis]  # Add channel dim
            processed.append(line)
        
        max_w = max(p.shape[1] for p in processed)
        batch = np.zeros((len(processed), 32, max_w, 1))
        for i, p in enumerate(processed):
            batch[i, :, :p.shape[1], 0] = p.squeeze()
        return batch.astype(np.float32)
    
    def ctc_decode(self, predictions):
        """Greedy CTC decoding"""
        texts = []
        for pred in predictions:
            pred_idx = tf.argmax(pred, axis=-1)
            text = self.greedy_ctc_decode(pred_idx.numpy())
            texts.append(text)
        return texts
    
    def greedy_ctc_decode(self, pred_idx):
        prev = self.blank_idx
        text = []
        for idx in pred_idx:
            if idx != self.blank_idx and idx != prev:
                text.append(self.idx_to_char[idx])
            prev = idx
        return ''.join(text)


## Step 3: Language Model Post-Processing

Simple n-gram based correction using a dictionary:

In [ ]:
class LanguageModel:
    def __init__(self, dictionary_path: str):
        self.words = set(open(dictionary_path).read().split())
        self.bigram_scores = self.load_bigrams('bigrams.txt')  # optional
    
    def correct(self, text: str) -> str:
        """Correct using dictionary + bigram scores"""
        words = text.split()
        corrected = []
        
        for word in words:
            # Exact match
            if word in self.words:
                corrected.append(word)
            else:
                # Try removing common OCR errors
                candidates = self.get_candidates(word)
                best = max(candidates, key=lambda w: self.word_score(w))
                corrected.append(best)
        
        return ' '.join(corrected)
    
    def get_candidates(self, word):
        candidates = [word]
        # Common substitutions: 0→O, 1→l/I, etc.
        subs = {'0': 'O', '1': 'l', '5': 'S', '|': 'I'}
        for i, c in enumerate(word):
            if c in subs:
                candidate = word[:i] + subs[c] + word[i+1:]
                if candidate in self.words:
                    candidates.append(candidate)
        return candidates
    
    def word_score(self, word):
        return 1.0 if word in self.words else 0.1


## Step 4: Complete Pipeline

In [ ]:
class TesseractInspiredOCR:
    def __init__(self, east_model_path, crnn_model_path, char_to_idx, dict_path):
        self.detector = EASTDetector(east_model_path)
        self.recognizer = CRNNRecognizer(crnn_model_path, char_to_idx)
        self.lm = LanguageModel(dict_path)
    
    def process_image(self, image_path: str) -> List[dict]:
        """Full pipeline: returns [{'bbox': (x1,y1,x2,y2), 'text': 'recognized text', 'confidence': 0.95}]"""
        image = cv2.imread(image_path)
        
        # 1. Text Detection
        boxes = self.detector.detect(image)
        
        # 2. Crop lines
        cropped_lines = [image[y1:y2, x1:x2] for x1,y1,x2,y2 in boxes]
        
        # 3. Recognition
        texts = self.recognizer.predict(cropped_lines)
        
        # 4. Post-process
        corrected_texts = [self.lm.correct(text) for text in texts]
        
        results = []
        for i, (box, text) in enumerate(zip(boxes, corrected_texts)):
            results.append({
                'bbox': box,
                'text': text,
                'confidence': 0.9  # placeholder, compute from CRNN probs
            })
        
        return results

# Usage
char_to_idx = {"a":1, "b":2, ...}  # your charset
ocr = TesseractInspiredOCR(
    east_model_path="frozen_east_text_detection.pb",
    crnn_model_path="crnn_model.keras",
    char_to_idx=char_to_idx,
    dict_path="english_words.txt"
)

results = ocr.process_image("document.jpg")
for res in results:
    print(f"Box {res['bbox']}: {res['text']}")


## Step 5 : Training Data Pipeline

For CRNN training data, generate synthetic line images:

In [ ]:
def generate_synthetic_lines(num_samples=10000, output_dir="synthetic_lines"):
    os.makedirs(output_dir, exist_ok=True)
    
    fonts = ["arial.ttf", "times.ttf"]  # etc.
    backgrounds = ["plain.png", "paper.png"]
    
    for i in range(num_samples):
        text = random_word()  # generate random text
        font = random.choice(fonts)
        bg = cv2.imread(random.choice(backgrounds))
        
        # Render text on background
        img = render_text_on_bg(text, font, bg)
        cv2.imwrite(f"{output_dir}/{i}.png", img)
        with open(f"{output_dir}/{i}.txt", 'w') as f:
            f.write(text)


What You Still Need

    Pre-trained EAST model: Download from official repo

    Trained CRNN: Use Keras captcha tutorial, train on synthetic data

    Dictionary file: Download english-words.txt

    Charset: Define your alphabet (a-zA-Z0-9 space punctuation)

This gives you a complete working Tesseract clone that you can train end-to-end. The EAST detector finds text lines, CRNN recognizes characters, language model corrects errors—just like Tesseract 4+.

In [ ]:
# Full OCR alphabet with special symbols
alphabet = """
abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ
0123456789
 #@$%=!?:;,+*-/()[]{}<>|"'`^_~&£€$¢¥•°×÷≈≠≤≥→←↑↓
""".strip().replace("\n", "")

# Remove any duplicates while preserving order
alphabet = "".join(dict.fromkeys(alphabet))  # 96 unique characters

print(f"Charset size: {len(alphabet)}")
print("Mapping preview:", dict(list(char_to_idx.items())[:10]))


char_to_idx = {char: idx + 1 for idx, char in enumerate(alphabet)}
idx_to_char = {idx: char for char, idx in char_to_idx.items()}
blank_idx = 0

charset_list = "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789 #@$%=!?:;,+*-/()[]{}<>|"'`^_~&£€¢°×÷≈"

### Keras CRNN Training Script (Ready-to-Run)

Here's the complete training script using your full charset:

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

# 1. Define your FULL charset
alphabet = "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789 #@$%=!?:;,+*-/()[]{}<>|"'`^_~&£€$¢¥•°×÷≈≠≤≥→←↑↓"
alphabet = "".join(dict.fromkeys(alphabet))  # deduplicate
n_classes = len(alphabet) + 1  # +1 for CTC blank

char_to_idx = {c: i + 1 for i, c in enumerate(alphabet)}
idx_to_char = {i + 1: c for i, c in enumerate(alphabet)}
blank_idx = 0

print(f"Charset: {len(alphabet)} characters")

# 2. CTC Layer (Keras 3 compatible)
class CTCLayer(layers.Layer):
    def __init__(self, name=None):
        super().__init__(name=name)
        self.loss_fn = keras.backend.ctc_batch_cost

    def call(self, y_pred, y_true, input_length, label_length):
        # Compute CTC batch cost
        batch_size = tf.shape(tf.expand_dims(label_length, 0))[0]
        max_label_len = tf.math.reduce_max(label_length)

        y_true = tf.nn.pad(y_true, [[0, 0], [0, max_label_len - tf.shape(y_true)[1]]], 'CONSTANT')
        y_pred = tf.nn.pad(y_pred, [[0, 0], [0, tf.shape(input_length)[0] - tf.shape(y_pred)[1]], [0, 0]], 'CONSTANT')

        loss = self.loss_fn(y_true, y_pred, input_length, label_length)
        self.add_loss(tf.reduce_mean(loss))
        return y_pred

# 3. CRNN Model
def build_crnn(input_shape=(32, None, 1), n_classes=97):
    inputs = keras.Input(shape=input_shape, name="input")
    
    # CNN Feature Extractor
    x = layers.Conv2D(32, (3, 3), activation="relu", padding="same")(inputs)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Conv2D(64, (3, 3), activation="relu", padding="same")(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Conv2D(128, (3, 3), activation="relu", padding="same")(x)
    x = layers.Conv2D(128, (3, 3), activation="relu", padding="same")(x)
    x = layers.MaxPooling2D((2, 1))(x)
    x = layers.Conv2D(256, (3, 3), activation="relu", padding="same")(x)
    x = layers.Conv2D(256, (3, 3), activation="relu", padding="same")(x)
    x = layers.MaxPooling2D((2, 1))(x)
    x = layers.Conv2D(512, (2, 2), activation="relu")(x)
    
    # Reshape to sequence [batch, timesteps, features]
    new_shape = ((x.shape[1] * x.shape[2]), x.shape[3])
    x = layers.Reshape(target_shape=new_shape, name="encode")(x)
    
    # Bidirectional LSTM
    x = layers.Bidirectional(layers.LSTM(256, return_sequences=True), name="bi-lstm-1")(x)
    x = layers.Bidirectional(layers.LSTM(256, return_sequences=True), name="bi-lstm-2")(x)
    
    # Output layer
    x = layers.Dense(n_classes, activation="softmax", name="dense2")(x)
    
    # Add CTC loss layer
    labels = keras.Input(name="label", shape=(None,), dtype="float32")
    input_length = keras.Input(name="input_length", shape=(), dtype="int64")
    label_length = keras.Input(name="label_length", shape=(), dtype="int64")
    
    output = CTCLayer(name="ctc_loss")(x, labels, input_length, label_length)
    
    return keras.Model(
        inputs=[inputs, labels, input_length, label_length], 
        outputs=output,
        name="ocr_model"
    )

# 4. Data Generator (synthetic for now)
def generate_synthetic_dataset(num_samples=50000, img_height=32):
    """Generate synthetic OCR training data"""
    samples = []
    
    # Sample texts covering all characters
    texts = []
    for length in [3, 5, 7, 9, 12]:
        for _ in range(num_samples // 20):
            text = np.random.choice(list(alphabet), length)
            texts.append("".join(text))
    
    for text in texts[:num_samples]:
        # Simulate image (replace with real data loading)
        width = max(100, len(text) * 30)
        img = np.random.rand(img_height, width, 1).astype(np.float32)
        label = np.array([char_to_idx[c] for c in text if c in char_to_idx])
        
        samples.append((img, label))
    
    return samples

# 5. Training
def main():
    # Build model
    model = build_crnn(n_classes=n_classes)
    model.summary()
    
    # Generate data (replace with your dataset)
    train_samples = generate_synthetic_dataset(20000)
    val_samples = generate_synthetic_dataset(5000)
    
    # Data generators
    BATCH_SIZE = 32
    
    def data_generator(samples):
        def gen():
            while True:
                np.random.shuffle(samples)
                for img, label in samples:
                    img_padded = np.pad(img, ((0,0),(0,128-img.shape[1]),(0,0)), mode='constant')
                    input_len = np.array([128//4])  # CNN reduces width by 4x
                    label_len = np.array([len(label)])
                    
                    yield (
                        {"input": img_padded[None,...], "label": label[None,...], 
                         "input_length": input_len, "label_length": label_len},
                        None  # No separate loss
                    )
        return gen
    
    train_ds = tf.data.Dataset.from_generator(
        data_generator(train_samples), 
        output_signature=(
            {
                "input": tf.TensorSpec(shape=(1, 32, 128, 1), dtype=tf.float32),
                "label": tf.TensorSpec(shape=(1, None), dtype=tf.int32),
                "input_length": tf.TensorSpec(shape=(), dtype=tf.int64),
                "label_length": tf.TensorSpec(shape=(), dtype=tf.int64)
            },
            None
        )
    ).batch(BATCH_SIZE).prefetch(8)
    
    # Compile and train
    model.compile(optimizer=keras.optimizers.Adam(1e-3))
    model.fit(train_ds, epochs=50, steps_per_epoch=len(train_samples)//BATCH_SIZE)
    
    # Save
    model.save("crnn_full_charset.keras")
    np.save("char_to_idx.npy", char_to_idx)
    np.save("idx_to_char.npy", idx_to_char)

if __name__ == "__main__":
    main()


### Usage with EAST Pipeline

Replace the CRNNRecognizer from the previous pipeline

In [ ]:
# Load trained model
model = keras.models.load_model("crnn_full_charset.keras", custom_objects={"CTCLayer": CTCLayer})
char_to_idx = np.load("char_to_idx.npy", allow_pickle=True).item()
recognizer = CRNNRecognizer(model, char_to_idx)  # from previous pipeline
